**Prova modificant encoder per la de torchrayvision**

In [ ]:
get_ipython().getoutput("pip install --upgrade pip")
get_ipython().getoutput("pip install segmentation-models-pytorch medmnist")



import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
from torchvision import transforms
from tqdm import tqdm
import segmentation_models_pytorch as smp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from medmnist import PneumoniaMNIST
import os
import gc

In [ ]:
!pip install torchxrayvision

In [ ]:
import torchxrayvision as xrv
modelo_temp = xrv.autoencoders.ResNetAE(weights="101-elastic")

# Llistem els noms de les capes internes reals
print("Components interns del model:")
for nom, capa in modelo_temp.named_children():
    print(f" - {nom}")

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import torchxrayvision as xrv

from medmnist import PneumoniaMNIST

# ==============================================================================
# 1. CONFIGURACIÓ GLOBAL
# ==============================================================================
CONFIG = {
    "IMG_SIZE": 224,
    "BATCH_SIZE": 16,
    "ACCUM_STEPS": 2,       # Simula batch de 32
    "LATENT_CHANNELS": 4,   # Latent de 4x28x28
    "EPOCHS_FASE_1": 20,    
    "EPOCHS_FASE_2": 80,    
    "LR_FASE_1": 1e-4,      
    "LR_FASE_2": 1e-5,      
    "VGG_WEIGHT": 0.05,
    "KL_WEIGHT": 0.00025,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "SAVE_DIR": "./artifacts_xrv"
}
os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)

# ==============================================================================
# 2. EL TURBODATASET (Carrega súper ràpida a GPU)
# ==============================================================================
class TurboDataset(Dataset):
    def __init__(self, data):
        # Normalitzem a 0-1 i pugem directament a la GPU!
        self.images = torch.tensor(data.imgs, dtype=torch.uint8).unsqueeze(1).float().div(255.0).to(CONFIG["DEVICE"])
        self.labels = torch.tensor(data.labels, dtype=torch.long).to(CONFIG["DEVICE"])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

def get_dataloaders():
    print(" Carregant dades de PneumoniaMNIST...")
    data_train = PneumoniaMNIST(split="train", download=True, size=CONFIG["IMG_SIZE"])
    data_val = PneumoniaMNIST(split="val", download=True, size=CONFIG["IMG_SIZE"])
    return {
        'train': DataLoader(TurboDataset(data_train), batch_size=CONFIG["BATCH_SIZE"], shuffle=True),
        'val':   DataLoader(TurboDataset(data_val), batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
    }

loaders = get_dataloaders()
print(f" Dades llestes! {len(loaders['train'].dataset)} imatges de train.")

# ==============================================================================
# 3. VGG PERCEPTUAL LOSS
# ==============================================================================
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in self.vgg.parameters():
            param.requires_grad = False

    def forward(self, recon, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
            recon = recon.repeat(1, 3, 1, 1)
        return nn.functional.l1_loss(self.vgg(recon), self.vgg(x))

vgg_loss = VGGPerceptualLoss().to(CONFIG["DEVICE"])

# ==============================================================================
# 4. CLASSES AUXILIARS (Early Stopping)
# ==============================================================================
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, current_loss):
        if self.best_loss is None:
            self.best_loss = current_loss
        elif current_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"  [Early Stopping] Sense millora... Paciència: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = current_loss
            self.counter = 0

# ==============================================================================
# 5. SONDA D'EXPLORACIÓ (TorchXRayVision)
# ==============================================================================
print("\n Llançant una sonda per detectar l'anatomia del model XRV...")
modelo_temp = xrv.autoencoders.ResNetAE(weights="101-elastic")
encoder_sonda = nn.Sequential(
    modelo_temp.conv1, modelo_temp.bn1, modelo_temp.relu, 
    modelo_temp.maxpool, modelo_temp.layer1, modelo_temp.layer2
)

dummy_x = torch.randn(1, 1, 224, 224)
dummy_out = encoder_sonda(dummy_x)
CANALES_XRV = dummy_out.shape[1]
RES_XRV = dummy_out.shape[2]
print(f" Sonda completada. Resolució: {RES_XRV}x{RES_XRV} amb {CANALES_XRV} canals.\n")

# ==============================================================================
# 6. ARQUITECTURA DEL MODEL HÍBRID
# ==============================================================================
class SpatialVAE_XRV(nn.Module):
    def __init__(self, canales_in, latent_channels=4):
        super().__init__()
        experto = xrv.autoencoders.ResNetAE(weights="101-elastic")
        
        self.encoder = nn.Sequential(
            experto.conv1, experto.bn1, experto.relu, 
            experto.maxpool, experto.layer1, experto.layer2
        )
        self.mu_conv = nn.Conv2d(canales_in, latent_channels, 1)       
        self.logvar_conv = nn.Conv2d(canales_in, latent_channels, 1)   
        
        self.decoder_input = nn.Conv2d(latent_channels, canales_in, 1) 
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(canales_in, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.BatchNorm2d(16), nn.ReLU(True),
            nn.Conv2d(16, 1, 3, 1, 1),
            nn.Sigmoid() 
        )

    def reparameterize(self, mu, log_var):
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(0.5 * log_var)
        return mu

    def forward(self, x):
        features = self.encoder(x)
        mu, log_var = self.mu_conv(features), self.logvar_conv(features)
        log_var = torch.clamp(log_var, -10, 10)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(self.decoder_input(z))
        return recon, mu, log_var

# ==============================================================================
# 7. INICIALITZACIÓ I FUNCIÓ D'ENTRENAMENT
# ==============================================================================
model = SpatialVAE_XRV(canales_in=CANALES_XRV, latent_channels=CONFIG["LATENT_CHANNELS"]).to(CONFIG["DEVICE"])
writer = SummaryWriter(log_dir=f"{CONFIG['SAVE_DIR']}/logs")
scaler = torch.cuda.amp.GradScaler()

def train_epoch(epoch, optimizer, loaders, model, scaler, CONFIG, writer):
    model.train()
    epoch_loss_total = epoch_loss_pixel = epoch_loss_vgg = epoch_loss_kl = 0
    
    optimizer.zero_grad(set_to_none=True) 
    loop = tqdm(loaders['train'], leave=False, desc=f"Epoch {epoch+1}")
    
    for i, (imgs, _) in enumerate(loop):
        # La classe TurboDataset ja puja imgs a la GPU, però ho deixem per seguretat
        imgs = imgs.to(CONFIG["DEVICE"]) 
        
        with torch.cuda.amp.autocast():
            recon, mu, log_var = model(imgs)
            loss_pixel = nn.L1Loss()(recon, imgs)
            loss_vgg = vgg_loss(recon, imgs) * CONFIG["VGG_WEIGHT"]
            loss_kl = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
            
            loss = loss_pixel + loss_vgg + (CONFIG["KL_WEIGHT"] * loss_kl)
            loss = loss / CONFIG["ACCUM_STEPS"]
        
        scaler.scale(loss).backward()
        
        if (i+1) % CONFIG["ACCUM_STEPS"] == 0 or (i+1) == len(loop):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        
        epoch_loss_total += loss.item() * CONFIG["ACCUM_STEPS"]
        epoch_loss_pixel += loss_pixel.item()
        epoch_loss_vgg += loss_vgg.item()
        epoch_loss_kl += loss_kl.item()
        
        loop.set_postfix(Loss=f"{loss.item()*CONFIG['ACCUM_STEPS']:.4f}")
        
    num_batches = len(loaders['train'])
    avg_loss_total = epoch_loss_total / num_batches
    print(f"Ep {epoch+1}: Total={avg_loss_total:.4f} | L1={epoch_loss_pixel/num_batches:.4f} | VGG={epoch_loss_vgg/num_batches:.4f} | KL={epoch_loss_kl/num_batches:.4f}")
    
    writer.add_scalar('Loss/Total', avg_loss_total, epoch)
    writer.add_scalar('Loss/Pixel_L1', epoch_loss_pixel/num_batches, epoch)
    writer.add_scalar('Loss/VGG_Perceptual', epoch_loss_vgg/num_batches, epoch)
    writer.add_scalar('Loss/KL_Divergence', epoch_loss_kl/num_batches, epoch)
    
    return avg_loss_total

# ==============================================================================
# 8. BUCLE D'ENTRENAMENT BIFÀSIC
# ==============================================================================

# --- FASE 1: CONGELAR ENCODER ---
print(f" INICIANT FASE 1: Congelant Encoder XRV ({CONFIG['EPOCHS_FASE_1']} Epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer_f1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG["LR_FASE_1"])
early_stopping = EarlyStopping(patience=15, min_delta=0.0001)

for epoch in range(CONFIG["EPOCHS_FASE_1"]):
    avg_loss = train_epoch(epoch, optimizer_f1, loaders, model, scaler, CONFIG, writer)
    
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f"{CONFIG['SAVE_DIR']}/vae_xrv_ep{epoch+1}.pth")
        torch.save(model.decoder.state_dict(), f"{CONFIG['SAVE_DIR']}/decoder_xrv_ep{epoch+1}.pth")
    
    early_stopping(avg_loss)
    if early_stopping.early_stop:
        print(" Early Stopping a la Fase 1!")
        break

# --- FASE 2: FINE-TUNING CONJUNT ---
print(f"\n INICIANT FASE 2: Descongelant tot per Fine-Tuning ({CONFIG['EPOCHS_FASE_2']} Epochs)")
for param in model.encoder.parameters():
    param.requires_grad = True

optimizer_f2 = optim.Adam(model.parameters(), lr=CONFIG["LR_FASE_2"])
early_stopping = EarlyStopping(patience=15, min_delta=0.0001) 

for epoch in range(CONFIG["EPOCHS_FASE_1"], CONFIG["EPOCHS_FASE_1"] + CONFIG["EPOCHS_FASE_2"]):
    avg_loss = train_epoch(epoch, optimizer_f2, loaders, model, scaler, CONFIG, writer)
    
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f"{CONFIG['SAVE_DIR']}/vae_xrv_ep{epoch+1}.pth")
        torch.save(model.decoder.state_dict(), f"{CONFIG['SAVE_DIR']}/decoder_xrv_ep{epoch+1}.pth")
    
    early_stopping(avg_loss)
    if early_stopping.early_stop:
        print(" Early Stopping a la Fase 2!")
        break

# ==============================================================================
# 9. GUARDAT FINAL I EXTRACCIÓ DE LATENTS
# ==============================================================================
torch.save(model.state_dict(), f"{CONFIG['SAVE_DIR']}/vae_xrv_final.pth")
torch.save(model.decoder.state_dict(), f"{CONFIG['SAVE_DIR']}/decoder_xrv_final.pth")
writer.close()

print("\n Guardant latents")
model.eval()
for split in ['train', 'val']: 
    latents, labels = [], []
    for x, y in tqdm(loaders[split], desc=f"Extraient {split}"):
        with torch.no_grad():
            _, mu, _ = model(x.to(CONFIG["DEVICE"]))
            latents.append(mu.cpu().numpy())
            labels.append(y.cpu().numpy())
    np.save(f"{CONFIG['SAVE_DIR']}/latents_xrv_{split}.npy", np.concatenate(latents))
    np.save(f"{CONFIG['SAVE_DIR']}/labels_xrv_{split}.npy", np.concatenate(labels))
    
print(" TOT COMPLETAT AMB ÈXIT!")

ELS 3 SUPERPLOTS!

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from medmnist import PneumoniaMNIST
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import os
import torchxrayvision as xrv #El necessitem perquè usem un nou model

# --- 1. CONFIGURACIÓ ---
CONFIG = {
    "IMG_SIZE": 224,
    "BATCH_SIZE": 32,  
    "LATENT_CHANNELS": 4,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "SAVE_DIR": "./artifacts_xrv", 
    "MODEL_PATH": "./artifacts_xrv/vae_xrv_final.pth", # Carreguem el nou model
    "LATENTS_PATH": "./artifacts_xrv/latents_xrv_train.npy" # Carreguem els latents train
}

# --- 2. DEFINICIÓ DEL MODEL (XRV- Expert en Radiografies) ---
class SpatialVAE_XRV(nn.Module):
    def __init__(self, canales_in=512, latent_channels=4): # De moment assumim 512 canals
        super().__init__()
        experto = xrv.autoencoders.ResNetAE(weights="101-elastic")
        
        # Tall a la layer2 per obtenir 28x28
        self.encoder = nn.Sequential(
            experto.conv1, experto.bn1, experto.relu, 
            experto.maxpool, experto.layer1, experto.layer2
        )
        self.mu_conv = nn.Conv2d(canales_in, latent_channels, 1)       
        self.logvar_conv = nn.Conv2d(canales_in, latent_channels, 1)   
        
        self.decoder_input = nn.Conv2d(latent_channels, canales_in, 1) 
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(canales_in, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.BatchNorm2d(16), nn.ReLU(True),
            nn.Conv2d(16, 1, 3, 1, 1),
            nn.Sigmoid() 
        )

    def forward(self, x):
        features = self.encoder(x)
        mu = self.mu_conv(features)
        recon = self.decoder(self.decoder_input(mu))
        return recon, mu, None

# --- 3. CARREGAR DADES ---
class TurboDataset(Dataset):
    def __init__(self, data):
        self.images = torch.tensor(data.imgs, dtype=torch.uint8).unsqueeze(1).float().div(255.0).to(CONFIG["DEVICE"])
        self.labels = torch.tensor(data.labels, dtype=torch.long).to(CONFIG["DEVICE"])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

def get_val_loader():
    print(" Carregant dades de validació...")
    data_val = PneumoniaMNIST(split="val", download=True, size=CONFIG["IMG_SIZE"])
    return DataLoader(TurboDataset(data_val), batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

# --- 4. GENERADOR DE GRÀFICS (Plots 1 y 2) ---
def recover_plots():
    if not os.path.exists(CONFIG["MODEL_PATH"]):
        print(f" ERROR: No trobo l'arxiu {CONFIG['MODEL_PATH']}")
        return

    print(f" Carregant model expert des de {CONFIG['MODEL_PATH']}...")
    model = SpatialVAE_XRV(canales_in=512, latent_channels=CONFIG["LATENT_CHANNELS"]).to(CONFIG["DEVICE"])
    state_dict = torch.load(CONFIG["MODEL_PATH"], map_location=CONFIG["DEVICE"])
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    loader = get_val_loader()

    # --- B. Plot de Reconstrucció ---
    print(" Generant comparativa visual...")
    imgs, labels = next(iter(loader))
    imgs = imgs.to(CONFIG["DEVICE"])

    try:
        idx_H = (labels == 0).nonzero(as_tuple=True)[0][0]
        idx_P = (labels == 1).nonzero(as_tuple=True)[0][0]
    except:
        idx_H, idx_P = 0, 1

    with torch.no_grad():
        rec_H, _, _ = model(imgs[idx_H].unsqueeze(0))
        rec_P, _, _ = model(imgs[idx_P].unsqueeze(0))

    fig, ax = plt.subplots(2, 3, figsize=(12, 8))
    
    def show(ax, img, title, diff=False):
        im = img.cpu().squeeze().numpy()
        cmap = 'inferno' if diff else 'gray'
        ax.imshow(im, cmap=cmap)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    show(ax[0,0], imgs[idx_H], "SA (Original)")
    show(ax[0,1], rec_H, "SA (Reconstrucció XRV)")
    show(ax[0,2], torch.abs(imgs[idx_H]-rec_H), "Error", diff=True)
    show(ax[1,0], imgs[idx_P], "PNEUMÒNIA (Original)")
    show(ax[1,1], rec_P, "PNEUMÒNIA (Reconstrucció XRV)")
    show(ax[1,2], torch.abs(imgs[idx_P]-rec_P), "Error", diff=True)

    plt.tight_layout()
    plt.show()

    # --- C. Plot PCA AMB DISTÀNCIA ---
    print("Calculant PCA i Distàncies...")
    latents, labels_list = [], []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i > 50: break 
            _, mu, _ = model(x)
            latents.append(mu.cpu().view(mu.size(0), -1).numpy())
            labels_list.append(y.cpu().numpy())
    
    X = np.concatenate(latents)
    y = np.concatenate(labels_list).squeeze() 

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    center_H = X_pca[y==0].mean(axis=0)
    center_P = X_pca[y==1].mean(axis=0)
    dist = np.linalg.norm(center_H - center_P)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(X_pca[y==0,0], X_pca[y==0,1], c='dodgerblue', alpha=0.4, label='Normal', s=20)
    plt.scatter(X_pca[y==1,0], X_pca[y==1,1], c='crimson', alpha=0.4, label='Pneumonia', s=20)
    plt.scatter(*center_H, c='navy', s=200, marker='X', edgecolors='white', linewidth=2, label='Centre Normal')
    plt.scatter(*center_P, c='darkred', s=200, marker='X', edgecolors='white', linewidth=2, label='Centre Pneumonia')
    plt.plot([center_H[0], center_P[0]], [center_H[1], center_P[1]], 'k--', alpha=0.5)

    plt.title(f"PCA Expert (4x28x28) | Distància entre Centres: {dist:.4f}", fontsize=14) 
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# --- 5. COMPROVACIO DE DISTANCIES (Plot 3) ---
def check_final_latents(file_path):
    print(f"\n Latents guardats a: {file_path}")
    try:
        data = np.load(file_path)
        data_flat = data.reshape(data.shape[0], -1)
        
        unique_data = np.unique(data_flat, axis=0)
        duplicates = len(data_flat) - len(unique_data)
        
        print(f"   Shape original: {data.shape}")
        print(f"   Duplicats exactes: {duplicates} (de {len(data)})")
        
        print("   Calculant distàncies knn...")
        nbrs = NearestNeighbors(n_neighbors=2, algorithm='ball_tree').fit(data_flat)
        distances, _ = nbrs.kneighbors(data_flat)
        
        nn_dists = distances[:, 1]
        mitjana = np.mean(nn_dists)
        minima = np.min(nn_dists[nn_dists > 0]) if len(nn_dists[nn_dists > 0]) > 0 else 0
        
        print(f"\n   RESULTATS DE DISTÀNCIA:")
        print(f"   Distància Mitjana entre Veïns: {mitjana:.4f} ")
        print(f"   Distància Mínima (no zero): {minima:.4f}")
        
        plt.figure(figsize=(8, 4))
        plt.hist(nn_dists, bins=50, color='mediumseagreen', edgecolor='black')
        plt.axvline(mitjana, color='red', linestyle='dashed', linewidth=2, label=f'Mitjana: {mitjana:.2f}')
        plt.title("Distribució de Distàncies a l'Espai Latent Final (4x28x28)") 
        plt.xlabel("Distància Euclidiana al Veí més proper")
        plt.ylabel("Freqüència (Pacients)")
        plt.legend()
        plt.show()

    except Exception as e:
        print(f" Error en llegir o processar l'arxiu: {e}")

if __name__ == "__main__":
    recover_plots()
    check_final_latents(CONFIG["LATENTS_PATH"])